## Missing Value Handling

In [1]:
# Ensure working directory is set to project root
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Import pandas for DataFrame missing value analysis and handling
import pandas as pd

# Load cleaned metadata CSV from previous data cleaning phase
cleaned_csv_path = "data/processed/utkface_cleaned.csv"
df = pd.read_csv(cleaned_csv_path)

print(f"Successfully loaded '{cleaned_csv_path}' ({len(df)} rows, {len(df.columns)} columns).")

Successfully loaded 'data/processed/utkface_cleaned.csv' (23707 rows, 5 columns).


In [2]:
# Calculate count and percentage of missing values per column
missing_count = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

# Construct a summary DataFrame for Before Treatment inspection
before_treatment_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing Percentage (%)': missing_percentage
})

print("=== BEFORE TREATMENT: MISSING VALUE ANALYSIS ===")
print(before_treatment_df)
print(f"\nTotal missing values across all columns: {df.isnull().sum().sum()}")

=== BEFORE TREATMENT: MISSING VALUE ANALYSIS ===
            Missing Count  Missing Percentage (%)
image_name              0                0.000000
age                     0                0.000000
gender                  1                0.004218
race                    1                0.004218
filepath                0                0.000000

Total missing values across all columns: 2


### Missing Value Treatment Decision Logic

The decision on missing value treatment depends entirely on the empirical findings from the **Before Treatment** check:
- **Empirical Finding:** Missing values were detected in `gender` and `race` categorical attributes.
- **Technical Rationale:** For discrete categorical variables (`gender` and `race`), missing entries are filled using **Mode Imputation** (the most frequent category in the dataset). This preserves record count without introducing out-of-range numerical artifacts.
- **Action Taken:** Mode Imputation is automatically applied to missing categorical entries, followed by post-treatment validation to confirm 0 missing values remain.

In [3]:
# Total missing values count across entire DataFrame
total_missing_values = df.isnull().sum().sum()

# Conditional treatment logic: Only apply fillna if missing values actually exist (> 0)
if total_missing_values > 0:
    print(f"Detected {total_missing_values} missing values. Applying appropriate treatment...")
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            if col == 'age':
                # Numerical feature: Impute with median to remain robust against skewed age distribution
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                print(f"Imputed column '{col}' with median: {median_val}")
            elif col in ['gender', 'race']:
                # Categorical feature: Impute with mode (most frequent discrete code)
                mode_val = df[col].mode()[0]
                df[col] = df[col].fillna(mode_val)
                print(f"Imputed column '{col}' with mode: {mode_val}")
else:
    print("No missing values detected. No treatment required.")

Detected 2 missing values. Applying appropriate treatment...
Imputed column 'gender' with mode: 0.0
Imputed column 'race' with mode: 0.0


In [4]:
# Missing Value Treatment: Mode Imputation for Categorical Attributes
df_handled = df.copy()

for col in df_handled.columns:
    if df_handled[col].isnull().sum() > 0:
        mode_val = df_handled[col].mode()[0]
        df_handled[col] = df_handled[col].fillna(mode_val)
        print(f"Applied Mode Imputation to '{col}' column (filled missing entries with mode = {mode_val}).")

# Ensure categorical columns are explicitly cast back to int64
df_handled["gender"] = df_handled["gender"].astype(int)
df_handled["race"] = df_handled["race"].astype(int)

# Verification after treatment
post_missing_counts = df_handled.isnull().sum()
post_missing_percent = (post_missing_counts / len(df_handled)) * 100

after_missing_df = pd.DataFrame({
    "Missing Count": post_missing_counts,
    "Missing Percentage (%)": post_missing_percent.round(4)
})

print("\n=== AFTER TREATMENT - VERIFICATION ===")
print(after_missing_df)
print(f"\nVerification Status: Passed ({post_missing_counts.sum()} missing values confirmed).")



=== AFTER TREATMENT - VERIFICATION ===
            Missing Count  Missing Percentage (%)
image_name              0                     0.0
age                     0                     0.0
gender                  0                     0.0
race                    0                     0.0
filepath                0                     0.0

Verification Status: Passed (0 missing values confirmed).


In [5]:
# Export DataFrame to data/processed/utkface_missing_handled.csv
output_path = "data/processed/utkface_missing_handled.csv"
df.to_csv(output_path, index=False)

print(f"DataFrame exported successfully to '{output_path}' ({len(df)} rows).")

DataFrame exported successfully to 'data/processed/utkface_missing_handled.csv' (23707 rows).
